# 22. Does a fixed noise target give real forgetting?

Notebook 21 showed that forgetting does not remove one task. It switches the whole model
off. After the request, the model answers the same class for every image, and all seven
tasks that were meant to survive fall to chance. That happened in all six completed runs,
so it is what the method does, not an accident of one seed.

This notebook tests the suspected cause and a one-line fix.

### The cause

The paper's Algorithm 2 draws a **new** random target at every step, then pushes the
generated weights toward it. Averaging over many fresh zero-centred targets does not pull
toward noise. It pulls toward the middle of those targets, and the middle of zero-centred
noise is zero. Weights near zero make the model ignore its input, so it returns one fixed
answer.

### The fix

Draw **one** target when the request starts and reuse it for every step. Then the closest
point is that one target, which is real noise, so the weights should end up noise-like
instead of near zero.

An older numerical check in this repository already pointed this way: the spread of the
output finished at 0.04 with ten fresh draws, and at 0.98 with a single fixed draw. A
spread near 1 is noise-like. A spread near 0 is collapse.

### What success looks like

The paper says an unlearned task should behave like a network that was never trained. So
this notebook measures a never-trained network as a reference, and asks whether the fixed
target lands closer to it.

| | input sensitivity | classes used | accuracy | other tasks |
| --- | --- | --- | --- | --- |
| A never-trained network | about 1 | several | about 10% | not applicable |
| What the study produces now | 0.00005 | 1 | exactly 10.0% | all at chance |
| What the fix should produce | near 1 | several | about 10% | mostly unharmed |

**Input sensitivity** is how much the model's ten output scores change when it is shown a
different image. Near zero means it is not reading the image at all.

### Four runs, one model

1. **Never trained**: the reference the paper describes.
2. **Study code**: `trainer.forget` exactly as your runs used it.
3. **Copied loop, fresh target**: the loop rewritten in this notebook, drawing fresh
   targets. This must reproduce run 2. If it does not, the rewrite is wrong and nothing
   below can be trusted.
4. **Copied loop, fixed target**: the same loop with the single change under test.

Run 3 exists only to prove the rewrite is faithful, so any difference in run 4 comes from
the noise target and not from the rewriting.

Nothing is written into the study. The repository stays at its pinned revision: the
changed loop lives in this notebook, not in `uncle/trainer.py`.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config, random_stream
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng
# The progress panel only draws bars, so it cannot change a result. It
# therefore sits outside the pinned implementation, in scripts/, and this
# checkout is detached to CODE_VERSION, which predates it. Try the checkout
# first, then main, then fall back to ordinary bars rather than failing.
sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError:
    try:
        tools = Path.cwd() / '_tools'
        tools.mkdir(exist_ok=True)
        subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', 'main'], check=True)
        (tools / 'dashboard.py').write_bytes(subprocess.check_output(
            ['git', '-C', str(REPO), 'show', 'origin/main:scripts/dashboard.py']))
        sys.path.insert(0, str(tools))
        from dashboard import Dashboard
        print('Progress panel fetched from main.')
    except Exception as problem:
        print('Progress panel unavailable, using plain bars:', problem)

        class Dashboard:
            """Just enough of the panel to keep the notebook running."""

            def __init__(self, *args, **kwargs): pass
            def __enter__(self): return self
            def __exit__(self, *args): return False
            def note(self, text): print(text, flush=True)

            def bar(self, total, description, unit='step'):
                from uncle.telemetry import progress_bar
                return progress_bar(total, description)

import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Settings

`SEED` and `FORGET_TASK` choose which completed run to start from. Everything else matches
the saved study and should be left alone, so the numbers line up with notebook 21.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

ORDER = "order_01"
SEED = 0                 # 0, 1, or 2: which completed run to start from
FORGET_TASK = "3"        # "3" or "14"

# How many forgetting steps to run in the long comparison. Your study used
# 100. At 100 steps the fixed target barely helped, which suggests the model
# never gets far enough to reach the noise and only manages the easy part,
# shrinking toward zero. Running much longer tests that idea: if it is right,
# the weights should stop shrinking and start matching the noise instead.
STEPS = 1000

# The faithfulness check only has to prove the copied loop matches the study
# code, so it runs short and cheap. The long runs below are the real test.
FAITHFULNESS_STEPS = 100

MEASURE_EVERY = 100      # how often to stop and measure during a long run

# Results are saved to R2, as in notebooks 20 and 21. A Colab runtime is
# deleted when the session ends and takes the local figure and tables with it.
# This write goes into a separate "diagnostics" folder and never touches the
# study's own files, so it cannot affect a future resume.
SAVE_TO_R2 = True

# Hand the GPU back when the notebook finishes, after the results are saved
# and checked, never before.
RELEASE_GPU_WHEN_DONE = True

if FORGET_TASK not in ("3", "14") or SEED not in (0, 1, 2):
    raise ValueError("The study only completed seeds 0-2 forgetting task 3 or 14.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Match the numerical settings the original runs used, so the generated
# weights come out identical rather than merely similar.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab Secrets
or environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the study and the source model

The checks refuse to continue if the code revision or the class split differs from the
saved study. The source model is the one that already learned all eight tasks.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())

if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
NEW_TASK = plan["new_tasks"][0]
CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=ORDER_TASKS, download=True)

# Source models are always kept in R2, which is why no retraining is needed.
source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
branch_job = next(j for j in ALL_JOBS
                  if j["id"] == f"branches/{ORDER}/seed_{SEED}/forget_{FORGET_TASK}/learn_{NEW_TASK}")
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")

REQUESTS = tuple(("learn", t) for t in ORDER_TASKS)
REQUESTS += tuple(tuple(op) for op in branch_job["operations"])
PROTECTED = [t for t in state["seen"] if t != FORGET_TASK]
print("Source model:", source_job["id"])
print("Forgetting task", FORGET_TASK, "| protecting", len(PROTECTED), "others")

## 5. The measurement

The same measurements as notebook 21, so the two notebooks can be compared directly.
`build(restore=False)` gives an untrained model, which is the reference the paper
describes.

In [ ]:
from torch.func import functional_call
from torch.utils.data import DataLoader

def inspect(hypernet, target, trainer, task, full=False):
    """Measure one task. With full=True also report how it uses its input.

    counts            how many of the 500 images landed in each class. One bar
                      of 500 means a single fixed answer.
    input_sensitivity the largest change in any of the ten output scores
                      between one image and another. Near zero means the model
                      returns the same scores whatever it is shown.
    """
    weights = hypernet.weights_for(task)
    buffers = {name: value.clone() for name, value in trainer.task_buffers[task].items()}
    loader = DataLoader(tasks[task]["test"], batch_size=EVAL_BATCH)
    picked, correct, total = [], 0, 0
    lowest = highest = None
    with torch.no_grad():
        for images, labels in loader:
            scores = functional_call(target, (weights, buffers),
                                     (images.to(trainer.device),), strict=True)
            choice = scores.argmax(1)
            picked.append(choice.cpu())
            correct += (choice == labels.to(trainer.device)).sum().item()
            total += labels.numel()
            if full:
                low, high = scores.min(dim=0).values, scores.max(dim=0).values
                lowest = low if lowest is None else torch.minimum(lowest, low)
                highest = high if highest is None else torch.maximum(highest, high)
    result = {"accuracy": 100.0 * correct / total}
    if full:
        counts = torch.bincount(torch.cat(picked), minlength=CLASSES).tolist()
        result["class_counts"] = counts
        result["distinct_classes"] = sum(1 for c in counts if c > 0)
        result["input_sensitivity"] = (highest - lowest).max().item()
    return result

def snapshot_row(hypernet, target, trainer, step, rows, quiet=False):
    """Record one row of evidence at this point in a run."""
    was_training = hypernet.training
    hypernet.eval(); target.eval()
    row = {"step": step, "weight_size": hypernet.raw_for(FORGET_TASK).norm().item()}
    row.update(inspect(hypernet, target, trainer, FORGET_TASK, full=True))
    row["other_tasks"] = {t: inspect(hypernet, target, trainer, t)["accuracy"]
                          for t in ORDER_TASKS if t != FORGET_TASK}
    rows.append(row)
    if not quiet:
        others = sum(row["other_tasks"].values()) / len(row["other_tasks"])
        print(f"    step {step:>3} | forgotten {row['accuracy']:5.1f}% | others avg {others:5.1f}% "
              f"| classes {row['distinct_classes']:>2} | sensitivity {row['input_sensitivity']:.3e}")
    if was_training:
        hypernet.train()
    return row

def build(restore=True):
    """A model built the way the study runner builds it.

    restore=False leaves it untrained, which is the reference the paper
    describes: a network that never learned this task.
    """
    config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE, "requests": REQUESTS})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=True)
    if restore:
        checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
        restore_rng(state["all_rng"])
    else:
        for task in ORDER_TASKS:
            hypernet.add_task(task)
        trainer.task_buffers = {t: dict(trainer.buffer_template) for t in ORDER_TASKS}
    return config, target, hypernet, trainer

## 6. The forgetting loop, with the noise target exposed

This is `uncle.trainer.forget` copied out, with one switch added. Read the comment inside:
it is the only difference between the published method and the proposed fix.

In [ ]:
def run_forgetting(trainer, task, protected, *, fixed_target, steps=None, on_step=None):
    """One forgetting request, copied from uncle.trainer.forget.

    Everything here matches the study code except the noise target, which is
    the whole point of this notebook:

    fixed_target=False  a new random target is drawn every step. This is what
                        the paper's Algorithm 2 says, and what your runs did.
                        Averaged over many fresh zero-centred targets, the
                        closest point is their middle, which is zero.

    fixed_target=True   one target is drawn at the start and reused for every
                        step. The closest point is then that target itself,
                        which is real noise rather than nothing.

    The repository is left untouched, so it stays at its pinned revision and
    the existing study results stay valid.
    """
    hypernet, config = trainer.hypernet, trainer.config

    # A frozen copy of the model as it is now. The protection term compares
    # the other tasks against this, so they are held where they started.
    snapshot = hypernet.snapshot()
    trainer._reference = {}          # preserve() caches here; start clean

    trainable = hypernet.generator_parameters()
    hypernet.requires_grad_(False)
    for parameter in trainable:
        parameter.requires_grad_(True)
    rate = config.forgetting_learning_rate
    optimizer = torch.optim.Adam(
        trainable, lr=config.learning_rate if rate is None else rate)
    hypernet.train()

    # The same noise stream the study code uses, addressed by seed, purpose and
    # task. Using it means the fresh-target run below reproduces the study's
    # numbers exactly rather than merely closely.
    noise = random_stream(config.seed, "forget", task, trainer.device)

    def draw(like):
        return torch.randn(like.shape, generator=noise,
                           device=like.device, dtype=like.dtype)

    target_noise, losses = None, []
    steps = config.burn_in if steps is None else steps
    if on_step is not None:
        with torch.no_grad():
            on_step(0)
        hypernet.train()

    for step in range(1, steps + 1):
        raw = hypernet.raw_for(task)
        if fixed_target:
            if target_noise is None:
                target_noise = draw(raw)          # drawn once, then reused
            to_noise = (raw - target_noise).square().sum()
        else:
            to_noise = sum((raw - draw(raw)).square().sum()
                           for _ in range(config.noise_samples)) / config.noise_samples

        # Unlearning weights the noise term by gamma and the protection term by
        # one, matching the paper's objective for unlearning.
        loss = config.gamma * to_noise + trainer.preserve(protected, snapshot)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if on_step is not None and step % MEASURE_EVERY == 0:
            # no_grad matters here: the study code wraps its callbacks the same
            # way, and without it every measurement would build a graph.
            with torch.no_grad():
                on_step(step)
            hypernet.train()
    return losses

## 7. Run all four

Each run rebuilds the model from the saved weights, so they all start from exactly the
same place. Run 3 exists to prove the copied loop matches the study code; if that check
fails, run 4 proves nothing.

In [ ]:
import gc

runs = {}

def measure_only(label):
    """Record a model without changing it, for the never-trained reference."""
    config, target, hypernet, trainer = build(restore=False)
    rows = []
    snapshot_row(hypernet, target, trainer, 0, rows)
    runs[label] = {"rows": rows, "losses": []}
    del trainer, hypernet, target
    gc.collect()

def forgetting_run(label, steps, *, fixed_target=None, use_study_code=False, every=None):
    """Rebuild from the saved weights, then forget, recording as it goes."""
    config, target, hypernet, trainer = build(restore=True)
    every = every or MEASURE_EVERY
    rows = []
    if use_study_code:
        # The real thing, straight out of uncle/trainer.py. It takes its own
        # step count, so the short faithfulness run matches the copied loop.
        snapshot_row(hypernet, target, trainer, 0, rows)
        losses = trainer.forget(FORGET_TASK, PROTECTED, burn_in=steps,
            on_step=lambda r: (snapshot_row(hypernet, target, trainer, r["step"], rows)
                               if r["step"] > 0 and r["step"] % every == 0 else None))
    else:
        losses = run_forgetting(trainer, FORGET_TASK, PROTECTED, steps=steps,
                                fixed_target=fixed_target,
                                on_step=lambda step: snapshot_row(hypernet, target, trainer, step, rows))
    runs[label] = {"rows": rows, "losses": losses, "steps": steps}
    del trainer, hypernet, target
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

with Dashboard(f"Long forgetting test: seed {SEED}, forget task {FORGET_TASK}") as board:
    overall = board.bar(5, "Runs", unit="run")

    board.note("measuring a never-trained network, the reference")
    print("\n=== 1. never trained (the paper's reference) ===")
    measure_only("never trained"); overall.update(1)

    # Two short runs whose only job is to prove the copied loop is faithful.
    board.note(f"faithfulness check: study code, {FAITHFULNESS_STEPS} steps")
    print(f"\n=== 2. study code, {FAITHFULNESS_STEPS} steps (faithfulness check) ===")
    forgetting_run("study code, short", FAITHFULNESS_STEPS, use_study_code=True,
                   every=max(FAITHFULNESS_STEPS // 5, 1)); overall.update(1)

    board.note(f"faithfulness check: copied loop, {FAITHFULNESS_STEPS} steps")
    print(f"\n=== 3. copied loop fresh, {FAITHFULNESS_STEPS} steps (must match run 2) ===")
    forgetting_run("copied fresh, short", FAITHFULNESS_STEPS, fixed_target=False,
                   every=max(FAITHFULNESS_STEPS // 5, 1)); overall.update(1)

    # The real question: does running much longer change where things end up?
    board.note(f"long run: fresh target, {STEPS} steps")
    print(f"\n=== 4. fresh target, {STEPS} steps (the published method, run long) ===")
    forgetting_run("fresh, long", STEPS, fixed_target=False); overall.update(1)

    board.note(f"long run: one fixed target, {STEPS} steps")
    print(f"\n=== 5. ONE fixed target, {STEPS} steps (the fix, run long) ===")
    forgetting_run("fixed, long", STEPS, fixed_target=True); overall.update(1)

    overall.close()
    board.note("all five runs measured")

# Is the rewritten loop faithful? If this fails, the long runs prove nothing.
a = runs["study code, short"]["rows"][-1]
b = runs["copied fresh, short"]["rows"][-1]
FAITHFUL = (abs(a["accuracy"] - b["accuracy"]) < 1e-9
            and abs(a["weight_size"] - b["weight_size"]) / max(a["weight_size"], 1e-9) < 1e-6)
print()
print(f"CHECK: does the copied loop reproduce the study code over {FAITHFULNESS_STEPS} steps?")
print(f"  study code    : accuracy {a['accuracy']:.2f}%, weight size {a['weight_size']:.4f}")
print(f"  copied, fresh : accuracy {b['accuracy']:.2f}%, weight size {b['weight_size']:.4f}")
print("  faithful:", FAITHFUL)
if not FAITHFUL:
    print("  The rewrite differs from the study code. Treat the long runs as unproven.")

## 8. Compare

One row per run. The question is whether the fixed target lands nearer the never-trained
reference than the study code does.

In [ ]:
reference = runs["never trained"]["rows"][-1]
chance = 100.0 / CLASSES

def final(label):
    run = runs[label]
    row = run["rows"][-1]
    others = list(row["other_tasks"].values())
    return {
        "run": label,
        "steps": run.get("steps", 0),
        "accuracy": row["accuracy"],
        "classes": row["distinct_classes"],
        "sensitivity": row["input_sensitivity"],
        "weights": row["weight_size"],
        "others_mean": sum(others) / len(others),
        "others_above_chance": sum(1 for v in others if v > chance + 0.05),
        "others_total": len(others),
    }

table = [final(label) for label in
         ("never trained", "study code, short", "copied fresh, short",
          "fresh, long", "fixed, long")]

print("WHAT EACH RUN ENDS UP WITH")
print(f"{'run':<21} {'steps':>6} {'forgotten':>10} {'classes':>8} {'sensitivity':>13}"
      f" {'weights':>10} {'others avg':>11} {'others ok':>10}")
print("-" * 97)
for r in table:
    print(f"{r['run']:<21} {r['steps']:>6} {r['accuracy']:9.1f}% {r['classes']:>8}"
          f" {r['sensitivity']:13.3e} {r['weights']:10.1f} {r['others_mean']:10.1f}% "
          f"{r['others_above_chance']:>5} of {r['others_total']}")

fixed = next(r for r in table if r["run"] == "fixed, long")
fresh = next(r for r in table if r["run"] == "fresh, long")
goal = reference["input_sensitivity"]

print()
print("THE QUESTION THIS RUN WAS BUILT TO ANSWER")
print("  At 100 steps the weights only shrank. If that is because the model never")
print("  gets far enough to match the noise, then running ten times longer should")
print("  turn it around: the weights should stop shrinking and sensitivity climb.")
print()

def trend(label):
    """Did the weights keep shrinking to the end, or turn back up?"""
    sizes = [row["weight_size"] for row in runs[label]["rows"]]
    lowest = min(sizes)
    return sizes[0], lowest, sizes[-1], sizes[-1] > lowest * 1.05

for label in ("fresh, long", "fixed, long"):
    start, lowest, end, rebounded = trend(label)
    print(f"  {label:<12} weights {start:8.1f} -> lowest {lowest:8.1f} -> end {end:8.1f}"
          f"   {'turned back up' if rebounded else 'still shrinking'}")

print()
print("VERDICT")
print(f"  A never-trained network has sensitivity {goal:.1f}. That is the goal.")
print(f"  Fresh target, {fresh['steps']} steps: {fresh['sensitivity']:.3e}"
      f"  ({100 * fresh['sensitivity'] / goal:.4f}% of the goal)")
print(f"  Fixed target, {fixed['steps']} steps: {fixed['sensitivity']:.3e}"
      f"  ({100 * fixed['sensitivity'] / goal:.4f}% of the goal)")
print()
recovering = fixed["sensitivity"] > goal * 0.1
responsive = fixed["classes"] > 1
near_chance = abs(fixed["accuracy"] - chance) < 5.0
if recovering and near_chance:
    print("  Running longer works. The model is responding to its input again while")
    print("  the forgotten task sits near chance, which is what the paper describes.")
    print("  The published 100-step budget is simply far too short.")
    if fixed["others_above_chance"] == fixed["others_total"]:
        print("  Every protected task stayed above chance too, so damage is contained.")
else:
    print("  Running longer does NOT fix it.")
    print(f"  After {fixed['steps']} steps the model is still at"
          f" {100 * fixed['sensitivity'] / goal:.4f}% of a never-trained network.")
    print("  So the problem is not the step budget and not the noise target.")
    print("  Aiming the generated weights at noise does not reach noise at all:")
    print("  shrinking them toward zero is the easy direction and the optimiser")
    print("  takes it. That is a limit of the method itself, not a setting.")

## 9. Chart

Left: the forgotten task. Middle: the average of the seven protected tasks, which is where
the real damage shows. Right: input sensitivity on a log scale, with the never-trained
reference marked.

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone

EXPORT = CACHE / "diagnostics" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT.mkdir(parents=True, exist_ok=True)

shown = [("fresh, long", "#b8322a"), ("fixed, long", "#245d99")]
figure, axes = plt.subplots(1, 3, figsize=(16, 4.6))

for label, colour in shown:
    rows = runs[label]["rows"]
    steps = [r["step"] for r in rows]
    axes[0].plot(steps, [r["accuracy"] for r in rows], color=colour, marker="o", label=label)
    others = [sum(r["other_tasks"].values()) / len(r["other_tasks"]) for r in rows]
    axes[1].plot(steps, others, color=colour, marker="o", label=label)
    axes[2].plot(steps, [max(r["input_sensitivity"], 1e-12) for r in rows],
                 color=colour, marker="o", label=label)

axes[0].axhline(100.0 / CLASSES, color="black", linestyle=":", linewidth=1)
axes[0].set_title(f"Forgotten task {FORGET_TASK}"); axes[0].set_ylabel("accuracy (%)")
axes[1].axhline(100.0 / CLASSES, color="black", linestyle=":", linewidth=1)
axes[1].set_title("The seven protected tasks, average"); axes[1].set_ylabel("accuracy (%)")
axes[2].axhline(reference["input_sensitivity"], color="green", linestyle="--", linewidth=1.2,
                label="never trained")
axes[2].set_yscale("log"); axes[2].set_title("Input sensitivity")
axes[2].set_ylabel("largest change in output score")
for axis in axes:
    axis.set_xlabel("forgetting step"); axis.legend(fontsize=8)

figure.suptitle(f"{STUDY_ID} | {ORDER} seed {SEED} | {STEPS} forgetting steps")
figure.tight_layout()
figure.savefig(EXPORT / "fixed_noise.png", dpi=180)
figure.savefig(EXPORT / "fixed_noise.svg")
plt.show()
print("Figure written to", EXPORT)

## 10. Save

Writes a JSON summary, a comparison table and a per-step table next to the figure, then
copies them to R2 under a diagnostics folder separate from the study's own files.

In [ ]:
import csv

summary = {
    "study_id": STUDY_ID, "code_version": CODE_VERSION, "order": ORDER, "seed": SEED,
    "forget_task": FORGET_TASK, "device": DEVICE, "classes_per_task": CLASSES,
    "steps": STEPS, "faithfulness_steps": FAITHFULNESS_STEPS,
    "study_burn_in": plan["config"]["burn_in"], "gamma": plan["config"]["gamma"],
    "beta": plan["config"]["beta"], "noise_samples": plan["config"]["noise_samples"],
    "copied_loop_reproduces_study_code": FAITHFUL,
    "never_trained_reference": reference,
    "comparison": table,
    "runs": {label: {"steps": run.get("steps", 0), "rows": run["rows"]}
              for label, run in runs.items()},
    "final_losses": {label: (run["losses"][-1] if run["losses"] else None)
                     for label, run in runs.items()},
    "reading": ("The study draws a new noise target every step, which pulls the weights "
                "toward zero and makes the model answer one class for every image. One "
                "fixed target should pull them toward noise instead. A never-trained "
                "network is the reference the paper describes."),
}
(EXPORT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

with (EXPORT / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(table[0]))
    writer.writeheader()
    writer.writerows(table)

with (EXPORT / "steps.csv").open("w", newline="", encoding="utf-8") as stream:
    columns = ["run", "steps", "step", "accuracy", "weight_size", "distinct_classes",
               "input_sensitivity"] + [f"task_{t}" for t in ORDER_TASKS if t != FORGET_TASK]
    writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    for label, run in runs.items():
        for row in run["rows"]:
            line = {"run": label, "steps": run.get("steps", 0), "step": row["step"],
                    "accuracy": row["accuracy"],
                    "weight_size": row["weight_size"],
                    "distinct_classes": row["distinct_classes"],
                    "input_sensitivity": row["input_sensitivity"]}
            line.update({f"task_{t}": v for t, v in row["other_tasks"].items()})
            writer.writerow(line)

print("Saved locally:")
for item in sorted(EXPORT.iterdir()):
    print("  ", item)

if SAVE_TO_R2:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    with Dashboard("Saving results") as board:
        files = sorted(EXPORT.iterdir())
        sending = board.bar(len(files), "Upload diagnostics")
        for item in files:
            sending.set_postfix_str(item.name)
            store.upload(item, destination + "/" + item.name)
            sending.update(1)
        sending.close()
    print("Uploaded to", destination)
    SAVED_LOCATION = destination
else:
    SAVED_LOCATION = str(EXPORT)
    print()
    print("SAVE_TO_R2 is off, so the only copy is inside this runtime.")

EXPECTED_FILES = ["comparison.csv", "fixed_noise.png", "fixed_noise.svg",
                  "steps.csv", "summary.json"]
SAVE_VERIFIED = sorted(item.name for item in EXPORT.iterdir()) == EXPECTED_FILES
print()
print("All expected files present:", SAVE_VERIFIED)

## 11. Hand the GPU back

Releases the Colab runtime once the work is done. It refuses to release if any expected
file is missing, so a failed save cannot cost you the run.

In [ ]:
if not SAVE_VERIFIED:
    print("Not releasing the GPU: some output files are missing.")
    print("Rerun section 10 before closing this runtime, or the results are lost.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results saved to:", SAVED_LOCATION)
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
    print("Not running in Colab, so there is no runtime to release.")
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Share that path.")
    import gc
    for name in ("runs", "state", "tasks"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## What to do with the output

Send me the table and verdict from section 8, and whether the faithfulness check in
section 7 passed.

**If the fixed target works**, you have two results rather than one. First, the published
method does not do what it claims, shown across six runs in notebook 21. Second, a single
line explains why, and changing it fixes the behaviour. That is a much stronger paper than
either half alone.

**If the fixed target does not work**, the collapse has another cause and we look again.
Either way the faithfulness check matters: if the copied loop did not reproduce the study
code, then nothing in run 4 can be trusted and that is the first thing to fix.

**What this still does not test.** Whether the forgotten information is really gone. A
model that answers randomly may still hold the knowledge, recoverable with a little
retraining. That is the separate study set up in notebooks 16 and 17.

**And it does not reproduce the paper.** This runs one seed, one order, beta 0.1 against
the paper's 0.01 for Tiny ImageNet, and a sequence of your own. It shows what the
algorithm does in your setting. Saying anything about the authors' reported numbers would
need their code and their configuration.